# XMCA-Net — Kaggle Environment Bootstrap & Hardware Verification

**Strict Research Requirement:**
- Target Hardware: Kaggle Dual NVIDIA T4 (2 × 16GB VRAM)
- Objective: Verify environment, dual GPU setup, CUDA tensor allocation, memory availability, dependency compatibility, and artifact persistence.
- **NO VLM WEIGHT DOWNLOADS OR RESEARCH INFERENCE IN THIS NOTEBOOK.**

In [ ]:
# 1. Python Environment & Platform Verification
import os
import sys
import platform
import shutil
from pathlib import Path

print("=== 1. System Platform & Python ===")
print(f"Python Version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Architecture: {platform.machine()}")
print(f"Working Directory: {os.getcwd()}")
print(f"Is Kaggle Environment: {'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')}")

In [ ]:
# 2. PyTorch & CUDA Verification
import torch

print("=== 2. PyTorch & CUDA Status ===")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemError("FATAL: CUDA is not available. Dual T4 GPU accelerator must be enabled in Kaggle settings.")

print(f"CUDA Version: {torch.version.cuda}")
print(f"cuDNN Version: {torch.backends.cudnn.version()}")
print(f"Device Count: {torch.cuda.device_count()}")

In [ ]:
# 3. Dual NVIDIA T4 GPU Verification
num_gpus = torch.cuda.device_count()
print(f"=== 3. GPU Device Verification (Found: {num_gpus}) ===")
assert num_gpus == 2, f"FATAL: Expected exactly 2 GPUs for dual-T4 execution, but found {num_gpus}."

for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    total_mem_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
    print(f"GPU {i}: {name} | Total Memory: {total_mem_gb:.2f} GB")
    assert "T4" in name, f"FATAL: Expected NVIDIA T4 on device {i}, found {name}"
    assert total_mem_gb >= 14.5, f"FATAL: Expected ~15-16GB VRAM on device {i}, found {total_mem_gb:.2f} GB"

print("Dual NVIDIA T4 GPUs successfully verified!")

In [ ]:
# 4. Multi-GPU Tensor Allocation & Memory Check
print("=== 4. Cross-GPU Allocation Test ===")
for i in range(2):
    device = torch.device(f"cuda:{i}")
    # Allocate a temporary test tensor (500 MB)
    x = torch.empty((125, 1000, 1000), dtype=torch.float32, device=device)
    allocated_mb = torch.cuda.memory_allocated(i) / (1024 ** 2)
    reserved_mb = torch.cuda.memory_reserved(i) / (1024 ** 2)
    print(f"Device {i} Allocation Test: {allocated_mb:.1f} MB allocated, {reserved_mb:.1f} MB reserved.")
    del x
    torch.cuda.empty_cache()

print("GPU memory allocation and cleanup succeeded on both devices.")

In [ ]:
# 5. Dependency Compatibility Verification
print("=== 5. Dependency Compatibility Check ===")
import transformers
import accelerate
import datasets
import pydantic
import yaml
import PIL
import sklearn

print(f"transformers: {transformers.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print(f"datasets:     {datasets.__version__}")
print(f"pydantic:     {pydantic.__version__}")
print(f"pyyaml:       {yaml.__version__}")
print(f"Pillow:       {PIL.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print("All core dependencies imported successfully.")

In [ ]:
# 6. Storage & Artifact Persistence Check
import json
from datetime import datetime, timezone

print("=== 6. Storage & Artifact Persistence ===")
working_dir = Path(".")
total, used, free = shutil.disk_usage(working_dir)
print(f"Disk Total: {total / (1024**3):.2f} GB")
print(f"Disk Used:  {used / (1024**3):.2f} GB")
print(f"Disk Free:  {free / (1024**3):.2f} GB")

report_dir = Path("artifacts/reports")
report_dir.mkdir(parents=True, exist_ok=True)
report_file = report_dir / "kaggle_bootstrap_verification.json"

verification_payload = {
    "status": "VERIFIED",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "gpu_count": torch.cuda.device_count(),
    "devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "pytorch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "disk_free_gb": round(free / (1024**3), 2)
}

with open(report_file, "w", encoding="utf-8") as f:
    json.dump(verification_payload, f, indent=2)

with open(report_file, "r", encoding="utf-8") as f:
    read_back = json.load(f)

assert read_back["status"] == "VERIFIED"
print(f"Persistence verification successful. Report written to {report_file}")

## Bootstrap Verification Complete

> **STOP CONDITION:** Do not proceed to model weight downloads (LLaVA, Qwen2-VL, InternVL) or research inference until subsequent tasks are authorized.